# Data Challenge — Face Occlusion (Colab training)

Phase 2 of the hybrid pipeline. Runs the real training on Colab GPU.

**Before running**: pick a GPU runtime — `Exécution → Modifier le type d'exécution → T4 GPU` (free).

**Pre-requisite** (one-time): in your Google Drive, create a folder containing:
- `crops.zip` (1.7 GB — upload via drive.google.com)
- `occlusion_datasets.zip` (zip of the 2 CSV files — see note below)

**Why zip the CSVs?** When uploading raw `.csv` files via drive.google.com, Drive may convert them to Google Sheets format (`.gsheet`), which can't be `cp`'d as regular files. Zipping prevents that. Alternative: upload the CSVs directly into the Colab file pane (sidebar → folder icon → upload) — only 1.3 MB total.

## 1. Clone the repo and install deps

In [ ]:
# Replace with your GitHub repo URL
REPO_URL = 'https://github.com/<your-username>/data-challenge-42.git'
REPO_NAME = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')

import os
if not os.path.isdir(REPO_NAME):
    !git clone $REPO_URL
%cd $REPO_NAME

In [ ]:
!pip -q install pandas pillow torch torchvision tqdm matplotlib
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Mount Google Drive and stage data on local SSD

**Why copy to local instead of reading from Drive directly?** Reading 100k small image files through mounted Drive is *extremely* slow (network round-trip per file). Copying the zip to `/content/` (local SSD) and unzipping takes ~2-3 min, then training reads at full disk speed.

This cell is **idempotent**: safe to re-run, won't re-unzip if `/content/crops` already exists.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Edit if your Drive folder lives elsewhere
DRIVE_FOLDER = '/content/drive/MyDrive/PRO_et_Mastere/data-challenge-42'

!ls -lh "$DRIVE_FOLDER"

In [ ]:
import os, glob, shutil

DATA_DIR = '/content/occlusion_datasets'
IMAGE_DIR = '/content/crops'

# --- IMAGES ---------------------------------------------------------------
if not os.path.isdir(IMAGE_DIR):
    !cp "$DRIVE_FOLDER/crops.zip" /content/crops.zip
    !unzip -q -o /content/crops.zip -d /content/
    # The zip's top-level dir may be 'Crop_224_5fp_100K' (organizer naming).
    # Find whichever top-level dir contains database1/database2/database3 and rename to /content/crops.
    if not os.path.isdir(IMAGE_DIR):
        for cand in glob.glob('/content/*'):
            if os.path.isdir(cand) and any(os.path.isdir(os.path.join(cand, f'database{i}')) for i in (1, 2, 3)):
                os.rename(cand, IMAGE_DIR)
                break

# --- CSVs -----------------------------------------------------------------
if not os.path.exists(f'{DATA_DIR}/train.csv'):
    os.makedirs(DATA_DIR, exist_ok=True)
    if os.path.exists(f'{DRIVE_FOLDER}/occlusion_datasets.zip'):
        !unzip -q -o "$DRIVE_FOLDER/occlusion_datasets.zip" -d /content/
    else:
        # Fallback: try to copy real .csv files if they exist on Drive (not .gsheet).
        for name in ('train.csv', 'test_students.csv'):
            src = f'{DRIVE_FOLDER}/occlusion_datasets/{name}'
            if os.path.exists(src):
                shutil.copy(src, f'{DATA_DIR}/{name}')

# --- Sanity check ---------------------------------------------------------
missing_csv = not os.path.exists(f'{DATA_DIR}/train.csv')
missing_img = not os.path.isdir(IMAGE_DIR)
if missing_csv:
    raise FileNotFoundError(
        f'{DATA_DIR}/train.csv not found. '
        'Either upload occlusion_datasets.zip to your Drive folder, '
        'OR upload train.csv + test_students.csv directly into the Colab file pane '
        f'under {DATA_DIR}/'
    )
if missing_img:
    raise FileNotFoundError(f'{IMAGE_DIR} not found — check crops.zip on Drive.')

print('train.csv:', os.path.exists(f'{DATA_DIR}/train.csv'))
print('test_students.csv:', os.path.exists(f'{DATA_DIR}/test_students.csv'))
print('crops/ subfolders:', os.listdir(IMAGE_DIR))

## 3. Train

Recommended starting point: `resnet50`, 8 epochs, batch 128, lr 3e-4. On a T4 expect ~10-15 min/epoch (full 100k train).

In [ ]:
!python scripts/train.py \
  --data-dir $DATA_DIR \
  --image-dir $IMAGE_DIR \
  --backbone resnet50 \
  --epochs 8 \
  --batch-size 128 \
  --lr 3e-4 \
  --num-workers 2

## 4. Generate the submission file

In [ ]:
!python scripts/infer.py \
  --data-dir $DATA_DIR \
  --image-dir $IMAGE_DIR \
  --checkpoint checkpoints/resnet50_best.pt \
  --backbone resnet50 \
  --output test_predictions.csv

In [ ]:
# Download submission to your local machine
from google.colab import files
files.download('test_predictions.csv')

# Also: copy the checkpoint back to Drive so you don't lose it when the session ends
!cp checkpoints/resnet50_best.pt "$DRIVE_FOLDER/resnet50_best.pt"

## 5. Ideas for the next iteration

- Bigger backbone: `efficientnet_b0` → `efficientnet_b3` (via `timm`).
- Test-time augmentation: average prediction on image and its horizontal flip.
- Ensemble: train 2-3 backbones, average their predictions.
- Synthetic augmentation for the high-occlusion tail (only 36 train samples > 0.5).
- Tune sampler / loss balance independently.